# Pré-processamento dos dados brutos

Este notebook realiza o carregamento, limpeza e filtragem dos dados de ocorrências criminais.

**Entrada:** Arquivos CSV brutos em `./input/`

**Saída:** `./output/maceio/filtered_maceio.csv` e `./output/arapiraca/filtered_arapiraca.csv`

## 1. Carregamento dos CSVs brutos e limpeza inicial
1. Concatena todos os arquivos CSV do diretório de entrada
2. Converte DATA_HORA_FATO para datetime
3. Converte LATITUDE e LONGITUDE de string (com vírgula) para float
4. Remove registros com data ou coordenadas inválidas
5. Exibe a distribuição dos tipos de crime (NATUREZA_FATO)

In [ ]:

import pandas as pd
import glob
from pathlib import Path


def load_all_csvs(input_dir: str) -> pd.DataFrame:
    pattern = f"{input_dir}/*.[cC][sS][vV]"
    files = glob.glob(pattern)
    dfs = []
    for f in files:
        dfs.append(pd.read_csv(f, delimiter=";", low_memory=False))
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        raise SystemExit(f"Input csv files in {input_dir} not possible to load")


def save(
    data: pd.DataFrame,
    output_path: str,
) -> pd.DataFrame:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(output_path, index=False, sep=";")
    print(f"Saved {len(data)} rows to {output_path}")


df = load_all_csvs("./input")

# Convert string DATA_HORA_FATO to datetime
df["DATA_HORA"] = pd.to_datetime(
    df["DATA_HORA_FATO"], format="%d/%m/%Y %H:%M:%S", errors="coerce"
)
df = df.dropna(subset="DATA_HORA", inplace=False)

# Convert string LATITUDE and LONGITUDE to float
df["LATITUDE"] = df["LATITUDE"].astype(str).str.replace(",", ".").astype(float)
df["LONGITUDE"] = df["LONGITUDE"].astype(str).str.replace(",", ".").astype(float)
df = df.dropna(subset=["LATITUDE", "LONGITUDE"])

# Group crimes type
group = df["NATUREZA_FATO"].value_counts()
display(group)

NATUREZA_FATO
ROUBO A TRANSEUNTE                             75220
ROUBO DE VEÍCULO (MOTO)                        20342
ROUBO OUTROS                                   13971
ROUBO DE VEÍCULO (DE PASSEIO)                   7667
ROUBO A CASA COMERCIAL                          6254
ROUBO A TRANSPORTE COLETIVO URBANO              5786
TENTATIVA DE ROUBO                              2736
ROUBO A RESIDÊNCIA                              2274
ROUBO A TRANSPORTE COLETIVO RODOVIÁRIO           977
ROUBO DE VEÍCULO (OUTROS)                        640
ROUBO DE CARGAS                                  636
ROUBO A CORRESPONDENTE BANCÁRIO                  243
EXTORSÃO                                         236
ARROMBAMENTO A BANCO (EXPLOSIVO)                 128
EXTORSÃO MEDIANTE SEQUESTRO                       59
ROUBO A CORREIOS                                  52
ROUBO A BANCO                                     26
ROUBO A CASA LOTÉRICA                             14
ARROMBAMENTO EM AGÊNCIA BANCÁRIA

## 2. Filtragem por tipo de crime

Seleciona 8 naturezas de roubo violento contra o patrimônio. A coluna `ROUBO_GROUP` é gerada apenas como apoio para análise exploratória — o pipeline de treino agrega todas as ocorrências em um único indicador binário desde o início.

Naturezas excluídas: `ROUBO OUTROS` (genérico), `ROUBO A CORRESPONDENTE BANCÁRIO` e `ROUBO A CORREIOS` (baixo volume, dinâmica espacial distinta).


In [ ]:
# Filtragem por tipo de crime e (opcional) mapeamento para grupos
# Após análise das contagens em pre-processing/cell 2, foram excluídas:
#   - ROUBO OUTROS (descrição genérica)
#   - ROUBO A CORRESPONDENTE BANCÁRIO (baixo volume, dinâmica distinta)
#   - ROUBO A CORREIOS (baixo volume, dinâmica distinta)
# A coluna ROUBO_GROUP é mantida apenas para análise exploratória futura;
# o pipeline a partir de pre-processing-3 já agrega tudo num único indicador binário.

list_crimes = [
    "ROUBO A TRANSEUNTE",
    "ROUBO A RESIDÊNCIA",
    "ROUBO DE VEÍCULO (MOTO)",
    "ROUBO DE VEÍCULO (DE PASSEIO)",
    "ROUBO DE VEÍCULO (OUTROS)",
    "ROUBO A TRANSPORTE COLETIVO RODOVIÁRIO",
    "ROUBO A TRANSPORTE COLETIVO URBANO",
    "ROUBO A CASA COMERCIAL",
]


print("Shape before selecting the specified robbery types: ", df.shape)
df_filtered = df.copy()
df_filtered = df_filtered[df_filtered["NATUREZA_FATO"].isin(list_crimes)]
print("Shape after selecting the specified robbery types: ", df_filtered.shape)


def map_roubo_group(natureza):
    """Mapeia naturezas em macro-categorias. Mantido para análise exploratória;
    o pipeline de treino agrega tudo em um único indicador binário."""
    s = str(natureza).upper()
    exact_map = {
        "ROUBO A TRANSEUNTE": "street_robbery",
        "ROUBO DE VEÍCULO (MOTO)": "vehicle_robbery",
        "ROUBO DE VEÍCULO (DE PASSEIO)": "vehicle_robbery",
        "ROUBO DE VEÍCULO (OUTROS)": "vehicle_robbery",
        "ROUBO A TRANSPORTE COLETIVO RODOVIÁRIO": "public_transport_robbery",
        "ROUBO A TRANSPORTE COLETIVO URBANO": "public_transport_robbery",
        "ROUBO A CASA COMERCIAL": "commercial_robbery",
        "ROUBO A RESIDÊNCIA": "residential_robbery",
    }
    return exact_map.get(s)


df_filtered["ROUBO_GROUP"] = df_filtered["NATUREZA_FATO"].apply(map_roubo_group)
df_filtered = df_filtered[df_filtered["ROUBO_GROUP"].notna()]
print("Shape after grouping: ", df_filtered.shape)
display(df_filtered["ROUBO_GROUP"].value_counts())


Shape before selecting the specified robbery types:  (137300, 30)
Shape after selecting the specified robbery types:  (119455, 30)
Shape after grouping and filtering (keeps only mapped groups): (119455, 31)


## 3. Separação por cidade e exportação

Filtra os dados por cidade (Maceió e Arapiraca) e salva em CSVs separados.

In [4]:
# Seleciona as colunas necessárias e separa por cidade
# Salva um CSV para cada cidade no diretório de saída

# Select only the desired columns
df_with_selected_columns = df_filtered[
    [
        "LONGITUDE",
        "LATITUDE",
        "ROUBO_GROUP",
        "DATA_HORA",
        "CIDADE_FATO",
    ]
]


df_arapiraca = df_with_selected_columns.query("CIDADE_FATO == 'Arapiraca'").reset_index(
    drop=True
)
save(df_arapiraca, output_path="./output/arapiraca/filtered_arapiraca.csv")

df_maceio = df_with_selected_columns.query("CIDADE_FATO == 'Maceió'").reset_index(
    drop=True
)
save(df_maceio, output_path="./output/maceio/filtered_maceio.csv")

Saved 17421 rows to ./output/arapiraca/filtered_arapiraca.csv
Saved 72787 rows to ./output/maceio/filtered_maceio.csv
